# DataChat — Exploratory Data Analysis

This notebook demonstrates the EDA pipeline behind the DataChat app. We load a dataset, profile it, generate visualizations, and prepare it for LLM querying.

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn langchain langchain-huggingface

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Style
plt.style.use('dark_background')
sns.set_palette("husl")

print("Libraries loaded ✓")

## 1. Load Dataset

Replace `your_data.csv` with any CSV file.

In [ ]:
# Generate sample dataset for demonstration
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'customer_id':  range(1, n+1),
    'age':          np.random.randint(18, 75, n),
    'tenure':       np.random.randint(0, 120, n),
    'monthly_charges': np.random.uniform(20, 150, n).round(2),
    'support_calls': np.random.poisson(2, n).clip(0, 15),
    'satisfaction': np.random.randint(1, 6, n),
    'region':       np.random.choice(['North','South','East','West'], n),
    'churn':        np.random.choice([0, 1], n, p=[0.73, 0.27])
})

# To load your own data:
# df = pd.read_csv('your_data.csv')

print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Data Profiling

In [ ]:
# Overview
print("=== DTYPES ===")
print(df.dtypes.to_string())
print(f"\n=== MISSING VALUES ===")
print(df.isnull().sum().to_string())
print(f"\n=== NUMERIC SUMMARY ===")
df.describe().round(2)

## 3. Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Feature Distributions', fontsize=14, y=1.02)
numeric_cols = ['age', 'tenure', 'monthly_charges', 'support_calls', 'satisfaction']

for i, col in enumerate(numeric_cols):
    ax = axes[i // 3][i % 3]
    df[col].hist(bins=25, ax=ax, color='#6366f1', edgecolor='#333', alpha=0.85)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')

# Categorical
ax = axes[1][2]
df['region'].value_counts().plot(kind='bar', ax=ax, color='#10b981', edgecolor='#333')
ax.set_title('Region', fontsize=11)
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../diagrams/eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved to diagrams/eda_distributions.png")

## 4. Correlation Analysis

In [ ]:
corr = df.select_dtypes(include='number').corr()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig('../diagrams/correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. LangChain Pandas Agent (DataChat Core)

This is the core of the DataChat app — a LangChain agent that interprets natural language and runs Pandas code.

In [ ]:
# This block requires: pip install langchain langchain-experimental langchain-huggingface
# and a HuggingFace API token: export HF_API_TOKEN=hf_xxx

"""
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_huggingface import HuggingFaceEndpoint
import os

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    huggingfacehub_api_token=os.environ["HF_API_TOKEN"],
    max_new_tokens=512,
    temperature=0.1,
)

agent = create_pandas_dataframe_agent(llm, df, verbose=True, allow_dangerous_code=True)

# Natural language queries
questions = [
    "What is the average monthly charges for customers who churned vs those who didn't?",
    "Which region has the highest churn rate?",
    "What is the correlation between tenure and churn?",
]

for q in questions:
    print(f"Q: {q}")
    result = agent.invoke(q)
    print(f"A: {result['output']}\n")
"""

print("Uncomment the block above and add HF_API_TOKEN to run the live LLM agent.")
print("The DataChat frontend HTML app is fully functional without this step.")

## Summary

| Feature | Implementation |
|---|---|
| CSV parsing | PapaParse (browser) / pandas (Python) |
| EDA | Automated column profiling + visualizations |
| NL querying | LangChain Pandas Agent + HuggingFace LLM |
| Charts | Chart.js (browser) / matplotlib (Python) |
| API | FastAPI (datachat.py) |